In [ ]:
#HW4 - CNN
#05.12.2025

In [1]:
import os
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, roc_auc_score
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages
from google.colab import drive

print("Tüm kütüphaneler yüklendi")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Tüm kütüphaneler yüklendi
PyTorch version: 2.9.0+cu126
CUDA available: True


In [2]:
drive.mount('/content/drive', force_remount=True)
data_dir = '/content/drive/MyDrive/CaltechTinySplit'

# Sonuçlar için klasör oluşturma
results_dir = '/content/drive/MyDrive/CNN_Results'
os.makedirs(results_dir, exist_ok=True)
cm_dir = os.path.join(results_dir, 'confusion_matrices')
history_dir = os.path.join(results_dir, 'training_histories')
os.makedirs(cm_dir, exist_ok=True)
os.makedirs(history_dir, exist_ok=True)

# GPU kontrolü - GPU kullandım çünkü işlemler uzun sürmekte
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

Mounted at /content/drive

 Device: cuda
   GPU: Tesla T4
   Memory: 15.83 GB


In [3]:
# ============================================================================
# HÜCRE 3: CUSTOM TRANSFORM VE DATA AUGMENTATION
# ============================================================================
class AddGaussianNoise(object): #Hocanın istediği custom transform fonksiyonu(Kendime özel)
    """Custom transform: Görüntüye Gaussian gürültü ekler"""
    def __init__(self, mean=0., std=0.05):
        self.std = std
        self.mean = mean

    def __call__(self, tensor):
        return tensor + torch.randn(tensor.size(), device=tensor.device) * self.std + self.mean

    def __repr__(self):
        return f"AddGaussianNoise(mean={self.mean}, std={self.std})"

# Train transformları (augmentation + custom transform)
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    AddGaussianNoise(mean=0., std=0.03),  # Custom transform
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Test/Val transformları (sadece normalizasyon)
transform_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

print(" Transform fonksiyonları hazır")
print(f"   Train transforms: {len(transform_train.transforms)} adım")
print(f"   Test transforms: {len(transform_test.transforms)} adım")

 Transform fonksiyonları hazır
   Train transforms: 5 adım
   Test transforms: 3 adım


In [ ]:
# ============================================================================
# HÜCRE 4: VERİ SETLERİNİ YÜKLEME
# ============================================================================
try:
    train_data = datasets.ImageFolder(
        root=os.path.join(data_dir, 'train'),
        transform=transform_train
    )
    val_data = datasets.ImageFolder(
        root=os.path.join(data_dir, 'val'),
        transform=transform_test
    )
    test_data = datasets.ImageFolder(
        root=os.path.join(data_dir, 'test'),
        transform=transform_test
    )

    num_classes = len(train_data.classes)
    class_names = train_data.classes

    print(" Veri setleri başarıyla yüklendi!")
    print(f"\n Dataset İstatistikleri:")
    print(f"   ├─ Sınıf sayısı: {num_classes}")
    print(f"   ├─ Train samples: {len(train_data)}")
    print(f"   ├─ Val samples: {len(val_data)}")
    print(f"   └─ Test samples: {len(test_data)}")
    print(f"\n  Sınıf isimleri: {class_names}")

except Exception as e:
    print(f"   HATA: Dosya yolu bulunamadı!")
    print(f"   Kontrol et: {data_dir}")
    print(f"   Hata detayı: {e}")

 Veri setleri başarıyla yüklendi!

 Dataset İstatistikleri:
   ├─ Sınıf sayısı: 9
   ├─ Train samples: 1346
   ├─ Val samples: 165
   └─ Test samples: 174

  Sınıf isimleri: ['Faces', 'Motorbikes', 'camera', 'cannon', 'cellphone', 'flamingo', 'hawksbill', 'ibis', 'pizza']


In [ ]:
#============================================================================
# HÜCRE 5: AKTİVASYON FONKSİYONLARI
# ============================================================================
class Swish(nn.Module):
    """Swish activation: x * sigmoid(x)"""
    def forward(self, x):
        return x * torch.sigmoid(x)

class Mish(nn.Module):
    """Mish activation: x * tanh(softplus(x))"""
    def forward(self, x):
        return x * torch.tanh(F.softplus(x))

print(" Aktivasyon fonksiyonları tanımlandı:")
print("   ├─ ReLU (PyTorch built-in)")
print("   ├─ Swish (custom)")
print("   └─ Mish (custom)")


 Aktivasyon fonksiyonları tanımlandı:
   ├─ ReLU (PyTorch built-in)
   ├─ Swish (custom)
   └─ Mish (custom)


In [ ]:
# ============================================================================
# HÜCRE 6: CNN MODEL MİMARİSİ
# ============================================================================
class OptimizedCNN(nn.Module):
    """
    CNN Model Architecture (Homework Fig. 1'e göre)
    - 4 Konvolüsyon bloğu
    - BatchNorm + Activation + MaxPool
    - Dropout regularization
    - 3 katmanlı classifier
    """
    def __init__(self, num_classes, activation='relu'):
        super().__init__()

        # Aktivasyon seçimi
        if activation == 'relu':
            act = nn.ReLU
        elif activation == 'swish':
            act = Swish
        elif activation == 'mish':
            act = Mish
        else:
            raise ValueError(f"Geçersiz aktivasyon: {activation}")

        # Feature extraction layers
        self.features = nn.Sequential(
            # Block 1: 3 -> 32
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            act(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            act(),
            nn.MaxPool2d(2),  # 224 -> 112

            # Block 2: 32 -> 64
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            act(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            act(),
            nn.MaxPool2d(2),  # 112 -> 56

            # Block 3: 64 -> 128
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            act(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            act(),
            nn.MaxPool2d(2),  # 56 -> 28

            # Block 4: 128 -> 256
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            act(),
            nn.MaxPool2d(2),  # 28 -> 14

            # Adaptive pooling
            nn.AdaptiveAvgPool2d((7, 7))  # 256 x 7 x 7
        )

        self.drop = nn.Dropout(0.4)

        # Classification layers
        self.classifier = nn.Sequential(
            nn.Linear(256 * 7 * 7, 512),
            nn.BatchNorm1d(512),
            act(),
            nn.Dropout(0.25),
            nn.Linear(512, 128),
            act(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.drop(x)
        x = self.classifier(x)
        return x

# Test: Model oluşturma ve parametre sayısını gösterme
test_model = OptimizedCNN(num_classes, activation='relu')
total_params = sum(p.numel() for p in test_model.parameters())
trainable_params = sum(p.numel() for p in test_model.parameters() if p.requires_grad)

print(" CNN Model tanımlandı")
print(f"\n Model Parametreleri:")
print(f"   ├─ Toplam: {total_params:,}")
print(f"   └─ Trainable: {trainable_params:,}")

del test_model  # Bellekten temizleme

 CNN Model tanımlandı

 Model Parametreleri:
   ├─ Toplam: 7,074,473
   └─ Trainable: 7,074,473


In [ ]:
# ============================================================================
# HÜCRE 7: YARDIMCI FONKSİYONLAR
# ============================================================================
def get_loaders(batch_size):
    """Batch size'a göre dinamik DataLoader oluşturur"""
    train_loader = DataLoader(
        train_data,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )
    val_loader = DataLoader(
        val_data,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )
    test_loader = DataLoader(
        test_data,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )
    return train_loader, val_loader, test_loader

def train_epoch(model, loader, criterion, optimizer):
    """Bir epoch eğitim yapar"""
    model.train()
    total_loss = 0
    for img, lbl in loader:
        img, lbl = img.to(device), lbl.to(device)
        optimizer.zero_grad()
        out = model(img)
        loss = criterion(out, lbl)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * img.size(0)
    return total_loss / len(loader.dataset)

def evaluate(model, loader):
    """Model performansını değerlendirir"""
    model.eval()
    all_preds, all_lbls, all_probs = [], [], []
    with torch.no_grad():
        for img, lbl in loader:
            img, lbl = img.to(device), lbl.to(device)
            out = model(img)
            probs = F.softmax(out, dim=1)
            _, preds = torch.max(out, 1)
            all_preds.extend(preds.cpu().numpy())
            all_lbls.extend(lbl.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    return np.array(all_preds), np.array(all_lbls), np.array(all_probs)

print("----Yardımcı fonksiyonlar tanımlandı------")

----Yardımcı fonksiyonlar tanımlandı------


In [ ]:
# ============================================================================
# HÜCRE 8: GÖRSELLEŞTİRME FONKSİYONLARI
# ============================================================================
def plot_confusion_matrix(cm, class_names, title, save_path=None):
    """Confusion matrix'i görselleştirir"""
    plt.figure(figsize=(10, 8))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=class_names,
        yticklabels=class_names,
        cbar_kws={'label': 'Count'}
    )
    plt.title(title, fontsize=16, fontweight='bold')
    plt.ylabel('True Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

class TrainingHistory:
    """Training ve validation loss'ları saklar"""
    def __init__(self):
        self.train_losses = []
        self.val_losses = []

    def add(self, train_loss, val_loss):
        self.train_losses.append(train_loss)
        self.val_losses.append(val_loss)

    def plot(self, title, save_path=None):
        plt.figure(figsize=(10, 6))
        epochs = range(1, len(self.train_losses) + 1)
        plt.plot(epochs, self.train_losses, 'b-', label='Train Loss', linewidth=2)
        plt.plot(epochs, self.val_losses, 'r-', label='Val Loss', linewidth=2)
        plt.title(title, fontsize=14, fontweight='bold')
        plt.xlabel('Epoch', fontsize=12)
        plt.ylabel('Loss', fontsize=12)
        plt.legend(fontsize=10)
        plt.grid(True, alpha=0.3)
        plt.tight_layout()

        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()

print("---- Görselleştirme fonksiyonları hazır-----")

---- Görselleştirme fonksiyonları hazır-----


In [ ]:
# ============================================================================
# HÜCRE 9: Uygulamaya Hazırlık, Yapılacak olan deneylerin/Testlerin hazırlanması toplamda 12 adet
# ============================================================================
experiment_list = []

# 1. Tüm kombinasyonlar (3 optimizer × 3 activation = 9 deney)
for opt in ['adam', 'rmsprop', 'sgd']:
    for act in ['relu', 'swish', 'mish']:
        experiment_list.append({
            'opt': opt,
            'act': act,
            'lr': 1e-3,
            'batch': 16
        })

# 2. Extra hiperparametre denemeleri (ödevde istenen "change LR and batch-size")
experiment_list.append({'opt': 'adam', 'act': 'mish', 'lr': 1e-4, 'batch': 16})  # Düşük LR
experiment_list.append({'opt': 'adam', 'act': 'mish', 'lr': 1e-3, 'batch': 32})  # Büyük batch
experiment_list.append({'opt': 'sgd',  'act': 'relu', 'lr': 1e-2, 'batch': 16})  # Yüksek LR

print(f" Deney planı hazır: Toplam {len(experiment_list)} deney\n")
print(" Deney Listesi:")
for i, exp in enumerate(experiment_list, 1):
    print(f"   {i:2d}. {exp['opt'].upper():8s} + {exp['act'].upper():6s} "
          f"| LR={exp['lr']:.0e} | Batch={exp['batch']}")


 Deney planı hazır: Toplam 12 deney

 Deney Listesi:
    1. ADAM     + RELU   | LR=1e-03 | Batch=16
    2. ADAM     + SWISH  | LR=1e-03 | Batch=16
    3. ADAM     + MISH   | LR=1e-03 | Batch=16
    4. RMSPROP  + RELU   | LR=1e-03 | Batch=16
    5. RMSPROP  + SWISH  | LR=1e-03 | Batch=16
    6. RMSPROP  + MISH   | LR=1e-03 | Batch=16
    7. SGD      + RELU   | LR=1e-03 | Batch=16
    8. SGD      + SWISH  | LR=1e-03 | Batch=16
    9. SGD      + MISH   | LR=1e-03 | Batch=16
   10. ADAM     + MISH   | LR=1e-04 | Batch=16
   11. ADAM     + MISH   | LR=1e-03 | Batch=32
   12. SGD      + RELU   | LR=1e-02 | Batch=16


In [ ]:
# ============================================================================
# HÜCRE 10: ANA EĞİTİM DÖNGÜSÜ - TÜMEL DEĞİŞKENLER
# ============================================================================
EPOCHS = 50
all_results = []
confusion_matrices = {}
training_histories = {}

print(f"  Eğitim Ayarları:")
print(f"   ├─ Epoch: {EPOCHS}")
print(f"   ├─ Scheduler: MultiStepLR (milestones=[20,40], gamma=0.3)")
print(f"   ├─ Loss: CrossEntropyLoss")
print(f"   └─ Device: {device}")
print("\n" + "="*80)
print(" ----EĞİTİM BAŞLIYOR!!---")
print("="*80)

  Eğitim Ayarları:
   ├─ Epoch: 50
   ├─ Scheduler: MultiStepLR (milestones=[20,40], gamma=0.3)
   ├─ Loss: CrossEntropyLoss
   └─ Device: cuda

 ----EĞİTİM BAŞLIYOR!!---


In [ ]:
# ============================================================================
# HÜCRE 11: ANA EĞİTİM DÖNGÜSÜ - DENEYLER
# ============================================================================
for i, exp in enumerate(experiment_list):
    opt_name = exp['opt']
    act_name = exp['act']
    lr_val = exp['lr']
    bs_val = exp['batch']

    exp_id = f"{opt_name}_{act_name}_lr{lr_val}_bs{bs_val}"

    print(f"\n{'='*80}")
    print(f">>> Deney {i+1}/{len(experiment_list)}: {opt_name.upper()} + {act_name.upper()}")
    print(f"    LR={lr_val} | Batch={bs_val}")
    print(f"{'='*80}")

    # 1. DataLoader'ları oluşturma
    train_loader, val_loader, test_loader = get_loaders(bs_val)

    # 2. Model oluşturma
    model = OptimizedCNN(num_classes, activation=act_name).to(device)
    criterion = nn.CrossEntropyLoss()

    # 3. Optimizer seçme
    if opt_name == 'adam':
        optimizer = optim.Adam(model.parameters(), lr=lr_val)
    elif opt_name == 'rmsprop':
        optimizer = optim.RMSprop(model.parameters(), lr=lr_val)
    elif opt_name == 'sgd':
        optimizer = optim.SGD(model.parameters(), lr=lr_val, momentum=0.9)

    # 4. Learning rate scheduler
    scheduler = optim.lr_scheduler.MultiStepLR(
        optimizer,
        milestones=[20, 40],
        gamma=0.3
    )

    # 5. Model checkpoint için değişkenler
    best_loss = float('inf')
    best_filename = os.path.join(results_dir, f'best_{exp_id}.pth')

    # 6. Training history tracker
    history = TrainingHistory()

    # 7. Eğitim döngüsü
    start_time = time.time()
    for epoch in range(EPOCHS):
        # Train
        train_loss = train_epoch(model, train_loader, criterion, optimizer)

        # Validation
        model.eval()
        val_running_loss = 0
        with torch.no_grad():
            for img, lbl in val_loader:
                img, lbl = img.to(device), lbl.to(device)
                out = model(img)
                val_running_loss += criterion(out, lbl).item() * img.size(0)
        val_loss = val_running_loss / len(val_loader.dataset)

        # History'ye kaydetme
        history.add(train_loss, val_loss)

        # LR scheduler step
        scheduler.step()

        # En iyi modeli kaydet (ModelCheckpoint benzeri)
        if val_loss < best_loss:
            best_loss = val_loss
            torch.save(model.state_dict(), best_filename)

        # Progress gösterme (HER 10 epoch)
        if (epoch + 1) % 10 == 0:
            current_lr = optimizer.param_groups[0]['lr']
            print(f"   Epoch {epoch+1:02d} -> Train: {train_loss:.4f} | "
                  f"Val: {val_loss:.4f} | LR: {current_lr:.6f}")

    training_time = time.time() - start_time

    # 8. Test - En iyi modeli yükleme
    print(f"\n   Loading best model and testing")
    model.load_state_dict(torch.load(best_filename))
    preds, lbls, probs = evaluate(model, test_loader)

    # 9. Metrikleri hesaplama
    acc = accuracy_score(lbls, preds)
    f1 = f1_score(lbls, preds, average='weighted')
    try:
        auc = roc_auc_score(lbls, probs, multi_class='ovr', average='weighted')
    except:
        auc = 0.0

    # 10. Confusion Matrix
    cm = confusion_matrices[exp_id] = confusion_matrix(lbls, preds)

    # CM'yi kaydetme
    cm_title = f"Confusion Matrix\n{opt_name.upper()} + {act_name.upper()} "
    cm_title += f"(LR={lr_val}, Batch={bs_val})"
    cm_path = os.path.join(cm_dir, f'cm_{exp_id}.png')
    plot_confusion_matrix(cm, class_names, cm_title, cm_path)

    # 11. Training history'yi kaydetme
    training_histories[exp_id] = history
    history_path = os.path.join(history_dir, f'history_{exp_id}.png')
    history_title = f"Training History: {opt_name.upper()} + {act_name.upper()}"
    history.plot(history_title, history_path)

    # 12. Sonuçları yazdırma
    print(f"\n   -SONUÇLAR!!-:")
    print(f"   ├─ Accuracy:  {acc:.4f}")
    print(f"   ├─ F1-Score:  {f1:.4f}")
    print(f"   ├─ AUC:       {auc:.4f}")
    print(f"   ├─ Best Val Loss: {best_loss:.4f}")
    print(f"   └─ Training Time: {training_time:.1f}s")

    # 13. Sonuçları listeye ekleme
    all_results.append({
        'optimizer': opt_name,
        'activation': act_name,
        'lr': lr_val,
        'batch': bs_val,
        'acc': acc,
        'f1': f1,
        'auc': auc,
        'exp_id': exp_id,
        'time': training_time
    })

    # 14. GPU memory temizle
    del model, optimizer, scheduler
    torch.cuda.empty_cache()

print("\n" + "="*80)
print("----TÜM DENEYLER TAMAMLANDI!!!!!!-----")
print("="*80)




>>> Deney 1/12: ADAM + RELU
    LR=0.001 | Batch=16
   Epoch 10 -> Train: 0.1915 | Val: 0.2789 | LR: 0.001000
   Epoch 20 -> Train: 0.2365 | Val: 0.3018 | LR: 0.000300
   Epoch 30 -> Train: 0.0950 | Val: 0.2499 | LR: 0.000300
   Epoch 40 -> Train: 0.0553 | Val: 0.2676 | LR: 0.000090
   Epoch 50 -> Train: 0.0298 | Val: 0.2795 | LR: 0.000090

   Loading best model and testing

   -SONUÇLAR!!-:
   ├─ Accuracy:  0.9598
   ├─ F1-Score:  0.9561
   ├─ AUC:       0.9991
   ├─ Best Val Loss: 0.2309
   └─ Training Time: 785.3s

>>> Deney 2/12: ADAM + SWISH
    LR=0.001 | Batch=16
   Epoch 10 -> Train: 0.2381 | Val: 0.2950 | LR: 0.001000
   Epoch 20 -> Train: 0.1369 | Val: 0.2122 | LR: 0.000300
   Epoch 30 -> Train: 0.0633 | Val: 0.2762 | LR: 0.000300
   Epoch 40 -> Train: 0.1180 | Val: 0.2348 | LR: 0.000090
   Epoch 50 -> Train: 0.0724 | Val: 0.2867 | LR: 0.000090

   Loading best model and testing

   -SONUÇLAR!!-:
   ├─ Accuracy:  0.9540
   ├─ F1-Score:  0.9498
   ├─ AUC:       0.9983
   ├─ B

In [ ]:
#============================================================================
# HÜCRE 12: SONUÇLARI GÖSTER - TABLO
# ============================================================================
print("\n" + "="*100)
print(f"{'Optimizer':<12} {'Activation':<12} {'LR':<10} {'Batch':<8} "
      f"{'Accuracy':<10} {'F1-Score':<10} {'AUC':<10}")
print("="*100)
for r in all_results:
    print(f"{r['optimizer']:<12} {r['activation']:<12} {r['lr']:<10} {r['batch']:<8} "
          f"{r['acc']:.4f}     {r['f1']:.4f}     {r['auc']:.4f}")
print("="*100)


Optimizer    Activation   LR         Batch    Accuracy   F1-Score   AUC       
adam         relu         0.001      16       0.9598     0.9561     0.9991
adam         swish        0.001      16       0.9540     0.9498     0.9983
adam         mish         0.001      16       0.9253     0.9219     0.9971
rmsprop      relu         0.001      16       0.9483     0.9469     0.9983
rmsprop      swish        0.001      16       0.9598     0.9585     0.9994
rmsprop      mish         0.001      16       0.9253     0.9202     0.9978
sgd          relu         0.001      16       0.9483     0.9474     0.9982
sgd          swish        0.001      16       0.9713     0.9695     0.9982
sgd          mish         0.001      16       0.9425     0.9384     0.9986
adam         mish         0.0001     16       0.9655     0.9626     0.9993
adam         mish         0.001      32       0.9425     0.9381     0.9984
sgd          relu         0.01       16       0.9195     0.9196     0.9967


In [ ]:
# ============================================================================
# HÜCRE 13: EN İYİ MODELLERİ BUL
# ============================================================================
# F1-Score'a göre sırala
sorted_results = sorted(all_results, key=lambda x: x['f1'], reverse=True)

print("\n EN İYİ 3 MODEL (F1-Score'a göre):")
print("-" * 100)
for idx, r in enumerate(sorted_results[:3], 1):
    print(f"\n{idx}. {r['optimizer'].upper()} + {r['activation'].upper()} "
          f"(LR={r['lr']}, Batch={r['batch']})")
    print(f"   ├─ Accuracy:  {r['acc']:.4f}")
    print(f"   ├─ F1-Score:  {r['f1']:.4f}")
    print(f"   ├─ AUC:       {r['auc']:.4f}")
    print(f"   └─ Time:      {r['time']:.1f}s")

# Accuracy'ye göre en iyi
best_acc = max(all_results, key=lambda x: x['acc'])
print(f"\n En Yüksek Accuracy: {best_acc['acc']:.4f} "
      f"({best_acc['optimizer'].upper()} + {best_acc['activation'].upper()})")

# AUC'ye göre en iyi
best_auc = max(all_results, key=lambda x: x['auc'])
print(f"En Yüksek AUC: {best_auc['auc']:.4f} "
      f"({best_auc['optimizer'].upper()} + {best_auc['activation'].upper()})")



 EN İYİ 3 MODEL (F1-Score'a göre):
----------------------------------------------------------------------------------------------------

1. SGD + SWISH (LR=0.001, Batch=16)
   ├─ Accuracy:  0.9713
   ├─ F1-Score:  0.9695
   ├─ AUC:       0.9982
   └─ Time:      603.3s

2. ADAM + MISH (LR=0.0001, Batch=16)
   ├─ Accuracy:  0.9655
   ├─ F1-Score:  0.9626
   ├─ AUC:       0.9993
   └─ Time:      613.6s

3. RMSPROP + SWISH (LR=0.001, Batch=16)
   ├─ Accuracy:  0.9598
   ├─ F1-Score:  0.9585
   ├─ AUC:       0.9994
   └─ Time:      587.4s

 En Yüksek Accuracy: 0.9713 (SGD + SWISH)
En Yüksek AUC: 0.9994 (RMSPROP + SWISH)


In [ ]:
# ============================================================================
# HÜCRE 14: PDF RAPOR OLUŞTURMA - Bu kısmı kendi raporuma ekleyebilmek için görselleştirme amacıyla oluşturdum
# Drive'a kaydediyor
# ============================================================================
print("\n PDF Rapor oluşturuluyor.!!")
pdf_path = os.path.join(results_dir, 'CNN_Homework_Report.pdf')

with PdfPages(pdf_path) as pdf:
    # ==================== Sayfa 1: Genel Sonuçlar Tablosu ====================
    fig, ax = plt.subplots(figsize=(14, 10))
    ax.axis('tight')
    ax.axis('off')

    table_data = [['Optimizer', 'Activation', 'LR', 'Batch', 'Accuracy', 'F1', 'AUC', 'Time(s)']]
    for r in all_results:
        table_data.append([
            r['optimizer'].upper(),
            r['activation'].upper(),
            f"{r['lr']:.0e}",
            str(r['batch']),
            f"{r['acc']:.4f}",
            f"{r['f1']:.4f}",
            f"{r['auc']:.4f}",
            f"{r['time']:.1f}"
        ])

    table = ax.table(
        cellText=table_data,
        cellLoc='center',
        loc='center',
        colWidths=[0.12, 0.12, 0.08, 0.08, 0.12, 0.12, 0.12, 0.10]
    )
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 2.5)

    # Header'ı vurgulama
    for i in range(8):
        table[(0, i)].set_facecolor('#4472C4')
        table[(0, i)].set_text_props(weight='bold', color='white')

    # En iyi 3'ü vurgulama
    for idx, r in enumerate(sorted_results[:3]):
        row_idx = all_results.index(r) + 1
        for col in range(8):
            table[(row_idx, col)].set_facecolor('#C6E0B4')

    plt.title('CNN Model Performance - All Experiments (T4 GPU)',
              fontsize=18, fontweight='bold', pad=20)
    pdf.savefig(fig, bbox_inches='tight')
    plt.close()

    # ==================== Sayfa 2: Top 3 Karşılaştırma ====================
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    metrics = ['acc', 'f1', 'auc']
    titles = ['Accuracy Comparison', 'F1-Score Comparison', 'AUC Comparison']

    for idx, (metric, title) in enumerate(zip(metrics, titles)):
        top3 = sorted_results[:3]
        names = [f"{r['optimizer'].upper()}\n+{r['activation'].upper()}" for r in top3]
        values = [r[metric] for r in top3]

        bars = axes[idx].bar(names, values, color=['#2E7D32', '#1976D2', '#F57C00'])
        axes[idx].set_title(title, fontweight='bold', fontsize=12)
        axes[idx].set_ylim([min(values) - 0.02, 1.0])
        axes[idx].grid(axis='y', alpha=0.3)

        # Değerleri yazma
        for bar, val in zip(bars, values):
            height = bar.get_height()
            axes[idx].text(bar.get_x() + bar.get_width()/2., height + 0.005,
                          f'{val:.4f}', ha='center', va='bottom',
                          fontweight='bold', fontsize=11)

    plt.suptitle('Top 3 Models Performance Comparison',
                 fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    pdf.savefig(fig, bbox_inches='tight')
    plt.close()

    # ==================== Sayfa 3+: Confusion Matrices ====================
    for r in all_results:
        exp_id = r['exp_id']
        cm = confusion_matrices[exp_id]

        fig, ax = plt.subplots(figsize=(10, 8))
        sns.heatmap(
            cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            ax=ax, cbar_kws={'label': 'Count'}
        )

        title = f"Confusion Matrix: {r['optimizer'].upper()} + {r['activation'].upper()}\n"
        title += f"LR={r['lr']}, Batch={r['batch']} | "
        title += f"Acc={r['acc']:.4f}, F1={r['f1']:.4f}, AUC={r['auc']:.4f}"
        ax.set_title(title, fontsize=14, fontweight='bold', pad=15)
        ax.set_ylabel('True Label', fontsize=12)
        ax.set_xlabel('Predicted Label', fontsize=12)

        plt.tight_layout()
        pdf.savefig(fig, bbox_inches='tight')
        plt.close()

    # ==================== Son Sayfa: Training Histories ====================
    fig, axes = plt.subplots(3, 1, figsize=(12, 14))
    for idx, r in enumerate(sorted_results[:3]):
        exp_id = r['exp_id']
        history = training_histories[exp_id]

        epochs = range(1, len(history.train_losses) + 1)
        axes[idx].plot(epochs, history.train_losses, 'b-',
                      label='Train Loss', linewidth=2)
        axes[idx].plot(epochs, history.val_losses, 'r-',
                      label='Val Loss', linewidth=2)

        title = f"#{idx+1}: {r['optimizer'].upper()} + {r['activation'].upper()} "
        title += f"(LR={r['lr']}, Batch={r['batch']})"
        axes[idx].set_title(title, fontsize=12, fontweight='bold')
        axes[idx].set_xlabel('Epoch', fontsize=10)
        axes[idx].set_ylabel('Loss', fontsize=10)
        axes[idx].legend(fontsize=9)
        axes[idx].grid(True, alpha=0.3)

    plt.suptitle('Training History - Top 3 Models',
                 fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout()
    pdf.savefig(fig, bbox_inches='tight')
    plt.close()

print(f" PDF Rapor oluşturuldu: Drive'dan ulaşabilirsiniz! {pdf_path}")


 PDF Rapor oluşturuluyor.!!
 PDF Rapor oluşturuldu: Drive'dan ulaşabilirsiniz! /content/drive/MyDrive/CNN_Results/CNN_Homework_Report.pdf


In [ ]:
# ============================================================================
# HÜCRE 15: TEXT RAPOR KAYDETME - Görselliğin arka planda olduğu test sonuçlarını net görme amaçlı oluşturdum
# ============================================================================
txt_path = os.path.join(results_dir, 'experiment_results.txt')
with open(txt_path, 'w', encoding='utf-8') as f:
    f.write("="*100 + "\n")
    f.write("CNN HOMEWORK - EXPERIMENTAL RESULTS\n")
    f.write("GPU: NVIDIA T4\n")
    f.write(f"Date: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write("="*100 + "\n\n")

    f.write(f"{'Optimizer':<12} {'Activation':<12} {'LR':<10} {'Batch':<8} "
            f"{'Accuracy':<10} {'F1-Score':<10} {'AUC':<10} {'Time(s)':<10}\n")
    f.write("-"*100 + "\n")
    for r in all_results:
        f.write(f"{r['optimizer']:<12} {r['activation']:<12} {r['lr']:<10} {r['batch']:<8} "
                f"{r['acc']:.4f}     {r['f1']:.4f}     {r['auc']:.4f}     {r['time']:.1f}\n")

    f.write("\n" + "="*100 + "\n")
    f.write("TOP 3 MODELS (by F1-Score):\n")
    f.write("="*100 + "\n")
    for idx, r in enumerate(sorted_results[:3], 1):
        f.write(f"\n{idx}. {r['optimizer'].upper()} + {r['activation'].upper()} "
                f"(LR={r['lr']}, Batch={r['batch']})\n")
        f.write(f"   Accuracy:  {r['acc']:.4f}\n")
        f.write(f"   F1-Score:  {r['f1']:.4f}\n")
        f.write(f"   AUC:       {r['auc']:.4f}\n")
        f.write(f"   Time:      {r['time']:.1f}s\n")

    f.write("\n" + "="*100 + "\n")
    f.write("ANALYSIS:\n")
    f.write("="*100 + "\n")
    f.write(f"Best Accuracy: {best_acc['acc']:.4f} "
            f"({best_acc['optimizer'].upper()} + {best_acc['activation'].upper()})\n")
    f.write(f"Best F1-Score: {sorted_results[0]['f1']:.4f} "
            f"({sorted_results[0]['optimizer'].upper()} + {sorted_results[0]['activation'].upper()})\n")
    f.write(f"Best AUC:      {best_auc['auc']:.4f} "
            f"({best_auc['optimizer'].upper()} + {best_auc['activation'].upper()})\n")

    # Optimizer analizi
    f.write("\n" + "-"*100 + "\n")
    f.write("OPTIMIZER ANALYSIS:\n")
    for opt in ['adam', 'rmsprop', 'sgd']:
        opt_results = [r for r in all_results if r['optimizer'] == opt]
        avg_f1 = np.mean([r['f1'] for r in opt_results])
        f.write(f"  {opt.upper()}: Avg F1-Score = {avg_f1:.4f}\n")

    # Activation analizi
    f.write("\n" + "-"*100 + "\n")
    f.write("ACTIVATION ANALYSIS:\n")
    for act in ['relu', 'swish', 'mish']:
        act_results = [r for r in all_results if r['activation'] == act]
        avg_f1 = np.mean([r['f1'] for r in act_results])
        f.write(f"  {act.upper()}: Avg F1-Score = {avg_f1:.4f}\n")

print(f"Text formatında tüm sonuçları görebilirsiniz!! ->Drive linkinden ulaşş!!! {txt_path}")

Text formatında tüm sonuçları görebilirsiniz!! ->Drive linkinden ulaşş!!! /content/drive/MyDrive/CNN_Results/experiment_results.txt
